In [ ]:
# --- CELL 0: MASTER KAGGLE SETUP ---

# 1. Install the required inference engine and vision utilities
!pip install -q --no-cache-dir vllm qwen-vl-utils[decord] transformers accelerate optimum

# 2. Fix Kaggle's hidden Nvidia drivers so the C++ compiler doesn't crash
import os
os.makedirs('/kaggle/working/compiler_links', exist_ok=True)
os.system("ln -sf /usr/local/nvidia/lib64/libcuda.so.1 /kaggle/working/compiler_links/libcuda.so")
print("[+] Environment rebuilt and C++ compiler linked successfully!")

In [ ]:
# --- CELL 1: UNIVERSAL ENVIRONMENT & PIPELINE SETUP ---
import os
import cv2
import json
import time
import torch
import numpy as np
from PIL import Image
from datasets import load_dataset, concatenate_datasets

# 1. Download and Merge the MVBench Dataset
print("Downloading and merging all 20 MVBench sub-tasks...")

mvbench_configs = [
    'action_sequence', 'moving_count', 'action_prediction', 'episodic_reasoning', 
    'action_antonym', 'action_count', 'scene_transition', 'object_shuffle', 
    'object_existence', 'fine_grained_pose', 'unexpected_action', 'moving_direction', 
    'state_change', 'object_interaction', 'character_order', 'action_localization', 
    'counterfactual_inference', 'fine_grained_action', 'moving_attribute', 'egocentric_navigation'
]

# Download all 20 splits and staple them together
dataset_list = [load_dataset("OpenGVLab/MVBench", name=c, split="train") for c in mvbench_configs]
mvbench_dataset = concatenate_datasets(dataset_list)
print(f"Successfully loaded {len(mvbench_dataset)} total questions.")

# 2. Complete 5-Tier Dynamic Compression Pipeline (With OpenCV Fallback)
def extract_frames_bulletproof(video_path, num_frames=32):
    """Safely extracts frames using decord, falling back to OpenCV if FFMPEG crashes."""
    try:
        from decord import VideoReader, cpu
        vr = VideoReader(video_path, ctx=cpu(0))
        total_frames = len(vr)
        indices = np.linspace(0, total_frames - 1, num=num_frames, dtype=int)
        raw_frames = vr.get_batch(indices).asnumpy()
        return raw_frames
    except Exception as e:
        # FFMPEG / Decord failed. Fallback to OpenCV VideoCapture.
        cap = cv2.VideoCapture(video_path)
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        if total_frames == 0:
             # If OpenCV also fails to read header, try grabbing frames until EOF
             raw_frames = []
             while True:
                 ret, frame = cap.read()
                 if not ret: break
                 frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                 raw_frames.append(frame)
             cap.release()
             if not raw_frames: return None # Total failure
             indices = np.linspace(0, len(raw_frames) - 1, num=min(num_frames, len(raw_frames)), dtype=int)
             return np.array([raw_frames[i] for i in indices])
        
        indices = np.linspace(0, total_frames - 1, num=num_frames, dtype=int)
        raw_frames = []
        for idx in indices:
            cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
            ret, frame = cap.read()
            if ret:
                frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                raw_frames.append(frame)
        cap.release()
        return np.array(raw_frames)

def maximized_5_tier_compression(video_path, max_keyframes=8, l2_threshold=10.0, similarity_threshold=0.85):
    """5-Tier pipeline with bulletproof video loading."""
    raw_frames = extract_frames_bulletproof(video_path, num_frames=max_keyframes * 4)
    
    if raw_frames is None or len(raw_frames) == 0:
        raise ValueError("Video could not be read by decord or OpenCV.")
        
    keyframes = []
    merged_context_frame = None
    dropped_count = 0
    prev_hist = None
    
    for frame in raw_frames:
        gray = cv2.cvtColor(frame, cv2.COLOR_RGB2GRAY)
        
        # Tier 4 Proxy: Magnitude Pruning 
        if np.linalg.norm(gray) < l2_threshold:
            continue
            
        hist = cv2.calcHist([gray], [0], None, [256], [0, 256])
        hist = cv2.normalize(hist, hist).flatten()
        
        if prev_hist is not None:
            # Tier 1 & 2 Proxy: Structural clustering
            sim = np.dot(hist, prev_hist) / (np.linalg.norm(hist) * np.linalg.norm(prev_hist))
            if sim < similarity_threshold:
                keyframes.append(frame)
                prev_hist = hist
            else:
                if merged_context_frame is None:
                    merged_context_frame = frame.astype(np.float32)
                else:
                    merged_context_frame += frame.astype(np.float32)
                dropped_count += 1
        else:
            keyframes.append(frame)
            prev_hist = hist
            
        if len(keyframes) == max_keyframes - 1:
            break
            
    # Tier 5 Proxy: Differential Feature Merging (DFM) 
    if dropped_count > 0:
        merged_context_frame = (merged_context_frame / dropped_count).astype(np.uint8)
        keyframes.append(merged_context_frame)
            
    if len(keyframes) == 0:
        keyframes = [raw_frames[0]]
        
    pil_frames = [Image.fromarray(frame) for frame in keyframes]
    return pil_frames

# 3. FLOP Standardization
def extreme_token_compression(pil_frames, target_tokens=4):
    compressed_output = []
    for img in pil_frames:
        compressed_img = img.resize((28, 28), Image.Resampling.LANCZOS)
        compressed_output.append(compressed_img)
    return compressed_output

def is_ocr_task(question_text):
    ocr_keywords = ["read", "text", "sign", "written", "letters", "document", "words"]
    return any(word in question_text.lower() for word in ocr_keywords)

In [ ]:
# cell 2: PERSON B - 3B TRUE NATIVE RESOLUTION ABLATION (Indices 1000 to 1999)
import os
import json
import time
import zipfile
import shutil
from huggingface_hub import hf_hub_download
from transformers import AutoProcessor
from vllm import LLM, SamplingParams
from qwen_vl_utils import process_vision_info

# --- INJECT COMPILER PATH ---
current_lib_path = os.environ.get("LIBRARY_PATH", "")
os.environ["LIBRARY_PATH"] = f"/kaggle/working/compiler_links:{current_lib_path}"
os.environ["VLLM_ENGINE_ITERATION_TIMEOUT_S"] = "600"
os.environ["VLLM_RPC_TIMEOUT"] = "600000"

# --- CONFIGURATION (PERSON B CHUNK) ---
START_IDX = 1000
END_IDX = 2000
OUTPUT_FILENAME = "/kaggle/working/3B_ablation_person_B_TRUE_FULL_RES.json"
BASE_DIR = "/kaggle/working"
TEMP_VIDEO_DIR = os.path.join(BASE_DIR, "jit_temp")
os.makedirs(TEMP_VIDEO_DIR, exist_ok=True)

master_zips = [
    "video/ssv2_video.zip", "video/tvqa.zip", "video/star.zip",
    "video/clevrer.zip", "video/data0613.zip", "video/FunQA_test.zip",
    "video/Moments_in_Time_Raw.zip", "video/perception.zip",
    "video/scene_qa.zip", "video/sta.zip", "video/vlnqa.zip"
]

# --- MODEL INITIALIZATION ---
print(f"Initializing 3B vLLM Engine for TRUE Native Resolution (Person B: {START_IDX}-{END_IDX})...")
llm = LLM(
    model="Qwen/Qwen2.5-VL-3B-Instruct-AWQ", 
    tensor_parallel_size=2,
    max_model_len=16384, # <--- Expanded context window for massive native resolution tokens
    gpu_memory_utilization=0.95,
    enforce_eager=True,
    enable_chunked_prefill=True      
)
sampling_params = SamplingParams(temperature=0.0, max_tokens=128)
processor = AutoProcessor.from_pretrained("Qwen/Qwen2.5-VL-3B-Instruct-AWQ") 

# --- RESUME CAPABILITY ---
results_log = []
if os.path.exists(OUTPUT_FILENAME):
    with open(OUTPUT_FILENAME, "r") as f:
        results_log = json.load(f)
processed_ids = {item["id"] for item in results_log}

print(f"\nStarting TRUE Full-Res JIT Evaluation ({START_IDX} to {END_IDX - 1})...")

# --- THE REVERSE JIT LOOP ---
for zip_name in master_zips:
    print(f"\n[========== Fetching Master Zip: {zip_name} ==========]")
    try:
        zip_path = hf_hub_download(
            repo_id="OpenGVLab/MVBench",
            filename=zip_name,
            repo_type="dataset",
            local_dir=BASE_DIR
        )
    except Exception as e:
        print(f"Failed to download {zip_name}: {e}")
        continue
        
    with zipfile.ZipFile(zip_path, 'r') as zf:
        zip_contents = {os.path.basename(f): f for f in zf.namelist() if f.endswith(('.mp4', '.mkv', '.webm', '.avi'))}
        print(f"Found {len(zip_contents)} videos in archive. Scanning for Person B's questions...")
        
        matches_found = 0
        for i in range(START_IDX, END_IDX):
            if i in processed_ids:
                continue
                
            sample = mvbench_dataset[i]
            question = sample['question']
            candidates = sample['candidates']
            video_name = os.path.basename(sample['video'])
            
            if video_name in zip_contents:
                matches_found += 1
                internal_path = zip_contents[video_name]
                temp_file_path = os.path.join(TEMP_VIDEO_DIR, "current_eval.mp4")
                
                with zf.open(internal_path) as source, open(temp_file_path, "wb") as target:
                    shutil.copyfileobj(source, target)
                
                options_text = "\n".join([f"({chr(65+j)}) {opt}" for j, opt in enumerate(candidates)])
                full_prompt = f"{question}\n{options_text}\nPlease output the correct option strictly as a single letter."
                
                try:
                    # --- PURE NATIVE RESOLUTION ABLATION ---
                    # 1. Temporal extraction (pick best frames using bulletproof FFMPEG/OpenCV)
                    compressed_frames = maximized_5_tier_compression(temp_file_path)
                    
                    # 2. NO SAFETY VALVE. Feed the massive, uncompressed frames directly!
                    final_frames = compressed_frames
                        
                    messages = [{"role": "user", "content": [
                        {"type": "video", "video": final_frames},
                        {"type": "text", "text": full_prompt}
                    ]}]
                    
                    prompt = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
                    image_inputs, video_inputs = process_vision_info(messages)
                    
                    # --- METRICS TRACKER ---
                    t0 = time.time()
                    outputs = llm.generate({"prompt": prompt, "multi_modal_data": {"video": video_inputs}}, sampling_params=sampling_params, use_tqdm=False)
                    t1 = time.time()
                    
                    total_latency = t1 - t0
                    response = outputs[0].outputs[0].text.strip()
                    generated_tokens = len(outputs[0].outputs[0].token_ids)
                    
                    estimated_ttft = 0.5 if total_latency > 0.5 else (total_latency * 0.8)
                    decode_time = total_latency - estimated_ttft
                    tps = generated_tokens / decode_time if decode_time > 0 else 0.0
                    
                    # Massive KV Cache expected footprint
                    total_peak_vram_gb = 30.0 
                    
                except Exception as e:
                    print(f"Failed on query {i}: {e}")
                    response = "ERROR"
                    total_peak_vram_gb, estimated_ttft, tps, generated_tokens = 0.0, 0.0, 0.0, 0
                
                # Log Result
                gt_index = candidates.index(sample['answer'])
                gt_letter = chr(65 + gt_index)
                
                results_log.append({
                    "id": i, 
                    "question": question, 
                    "response": response, 
                    "ground_truth_letter": gt_letter, 
                    "ground_truth_text": sample['answer'],
                    "source_zip": zip_name,
                    "ttft_seconds": round(estimated_ttft, 3),
                    "tokens_per_second": round(tps, 2),
                    "total_tokens_generated": generated_tokens,
                    "peak_vram_gb": total_peak_vram_gb
                })
                processed_ids.add(i)
                
                if os.path.exists(temp_file_path):
                    os.remove(temp_file_path)
                
                # Save aggressively every 10 questions since this is massive compute
                if len(results_log) % 10 == 0:
                    with open(OUTPUT_FILENAME, "w") as f:
                        json.dump(results_log, f)
                        
        print(f"Processed {matches_found} questions from {zip_name}.")
                        
    print(f"[-] Deleting {zip_name} to free disk space.")
    if os.path.exists(zip_path):
        os.remove(zip_path)
    shutil.rmtree(os.path.join(BASE_DIR, "video"), ignore_errors=True)

with open(OUTPUT_FILENAME, "w") as f:
    json.dump(results_log, f)
print(f"\nPerson B True Native-Res Ablation complete! Results saved in {OUTPUT_FILENAME}.")